실습 4-B: 공격 키워드 빈도 분석 (터미널 / Jupyter 버전)
======================================================
1교시에서 배운 공격 키워드가 실제 CSIC 2010 데이터에
얼마나 나타나는지 확인합니다.

Streamlit 없이 터미널에서 바로 실행할 수 있는 버전입니다.
(Streamlit 대시보드 버전: streamlit run eda_visualization.py)

실행 방법 (둘 다 가능):
  - 스크립트 모드:    python eda_keyword_analysis.py
  - Jupyter 셀 모드:  VSCode에서 # %% 셀 위 'Run Cell' 클릭 (또는 Shift+Enter)


In [2]:
# %% [Setup] 패키지 import 및 데이터 로드
import pandas as pd
import os
import re
from urllib.parse import unquote

print("=" * 65)
print("  [실습 4-B] 공격 키워드 빈도 분석")
print("=" * 65)

# Jupyter 셀 모드에서도 동작하도록 __file__ 처리
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()

data_path = os.path.join(SCRIPT_DIR, "csic2010_requests.csv")
if not os.path.exists(data_path):
    print(f"\n  !! csic2010_requests.csv 파일이 없습니다.")
    print(f"  >> 먼저 download_csic2010.py를 실행하세요.")
    raise SystemExit

df = pd.read_csv(data_path)
print(f"\n  데이터 로드: {len(df):,}건")

# URL 디코딩 + 결합 텍스트
df["url_decoded"] = df["url"].apply(lambda x: unquote(str(x), encoding="latin-1"))
df["body_decoded"] = df["body"].fillna("").apply(lambda x: unquote(str(x), encoding="latin-1"))
df["full_text"] = df["url_decoded"] + " " + df["body_decoded"]


  [실습 4-B] 공격 키워드 빈도 분석

  데이터 로드: 10,000건


In [5]:
df.tail(20)

,label,method,url,http_version,host,user_agent,pragma,cache_control,accept,accept_encoding,...,cookie,content_type,content_length,connection,body,classification,raw_request,url_decoded,body_decoded,full_text
9980,Normal,POST,/tienda1/publico/entrar.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=6A9E8962E0E31D4B3FCCC24A3AF86C3B,application/x-www-form-urlencoded,Content-Length: 33,Connection: close,errorMsg=Credenciales+incorrectas,0,POST /tienda1/publico/entrar.jsp HTTP/1.1\nHos...,/tienda1/publico/entrar.jsp,errorMsg=Credenciales+incorrectas,/tienda1/publico/entrar.jsp errorMsg=Credencia...
9981,Normal,POST,/tienda1/publico/vaciar.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=862C13AF8D570864AC2AAF0BD8B61339,application/x-www-form-urlencoded,Content-Length: 17,Connection: close,B2=Vaciar+carrito,0,POST /tienda1/publico/vaciar.jsp HTTP/1.1\nHos...,/tienda1/publico/vaciar.jsp,B2=Vaciar+carrito,/tienda1/publico/vaciar.jsp B2=Vaciar+carrito
9982,Normal,GET,/tienda1/global/titulo.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=F37E996B0F96713381B14A43E1AE5DC7,NaN,NaN,close,NaN,0,GET /tienda1/global/titulo.jsp HTTP/1.1\nHost:...,/tienda1/global/titulo.jsp,,/tienda1/global/titulo.jsp
9983,Anomalous,POST,/tienda1/miembros/editar.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=40FF98C7374C2495311F834D6E9B3C70,application/x-www-form-urlencoded,Content-Length: 246,Connection: close,modo=registro&login=keefer&password=ma2e32&nom...,1,POST /tienda1/miembros/editar.jsp HTTP/1.1\nHo...,/tienda1/miembros/editar.jsp,modo=registro&login=keefer&password=ma2e32&nom...,/tienda1/miembros/editar.jsp modo=registro&log...
9984,Normal,GET,/tienda1/miembros/fotos.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=4F8E28F643212BD1A8E725B3ED26C4A1,NaN,NaN,close,NaN,0,GET /tienda1/miembros/fotos.jsp HTTP/1.1\nHost...,/tienda1/miembros/fotos.jsp,,/tienda1/miembros/fotos.jsp
9985,Normal,POST,/tienda1/publico/vaciar.jsp,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=B3C81D5FEF9DD87B4F9EA0BAD8858358,application/x-www-form-urlencoded,Content-Length: 17,Connection: close,B2=Vaciar+carrito,0,POST /tienda1/publico/vaciar.jsp HTTP/1.1\nHos...,/tienda1/publico/vaciar.jsp,B2=Vaciar+carrito,/tienda1/publico/vaciar.jsp B2=Vaciar+carrito
9986,Normal,GET,/tienda1/miembros/editar.jsp?modo=registro&log...,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=7866EF9DEBDA21FC507499CA39DA6554,NaN,NaN,close,NaN,0,GET /tienda1/miembros/editar.jsp?modo=registro...,/tienda1/miembros/editar.jsp?modo=registro&log...,,/tienda1/miembros/editar.jsp?modo=registro&log...
9987,Anomalous,GET,/tienda1/publico/caracteristicas.jsp.OLD,HTTP/1.1,localhost:8080,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate",...,JSESSIONID=73C4B5C89CE06CFC7365EBB7CF302DC2,NaN,NaN,close,NaN,1,GET /tienda1/publico/caracteristicas.jsp.OLD H...,/tienda1/publico/caracteristicas.jsp.OLD,,/tiend

In [6]:
# %% [1] 공격 키워드 카테고리별 탐지
# ============================================================
# 1. 공격 키워드 카테고리별 탐지
# ============================================================
print(f"\n{'─' * 65}")
print("  [1] 공격 키워드 카테고리별 빈도")
print(f"{'─' * 65}")

attack_categories = {
    "SQL Injection": ["'", "select", "union", "drop", "insert",
                      "delete", "update", "or '", "1=1", "--"],
    "XSS": ["<script", "alert(", "onerror", "<iframe",
            "<img", "javascript:", "onfocus"],
    "Path Traversal": ["../", "..\\", "/etc/passwd", "/etc/shadow"],
    "Command Injection": ["; ", "|", "&&", "/bin/", "cat ", "rm ", "wget"],
    "CRLF Injection": ["%0d", "%0a"],
}

print(f"\n  {'카테고리':<20s} | {'정상':>8s} | {'공격':>8s} | {'공격 집중도':>10s}")
print(f"  {'─' * 20}-+-{'─' * 8}-+-{'─' * 8}-+-{'─' * 10}")

normal_text = df[df["label"] == "Normal"]["full_text"]
attack_text = df[df["label"] == "Anomalous"]["full_text"]

for category, keywords in attack_categories.items():
    normal_count = 0
    attack_count = 0
    for kw in keywords:
        normal_count += normal_text.str.contains(re.escape(kw), case=False, na=False).sum()
        attack_count += attack_text.str.contains(re.escape(kw), case=False, na=False).sum()

    total = normal_count + attack_count
    concentration = attack_count / max(total, 1) * 100
    print(f"  {category:<20s} | {normal_count:>8,} | {attack_count:>8,} | {concentration:>9.1f}%")



─────────────────────────────────────────────────────────────────
  [1] 공격 키워드 카테고리별 빈도
─────────────────────────────────────────────────────────────────

  카테고리                 |       정상 |       공격 |     공격 집중도
  ────────────────────-+-────────-+-────────-+-──────────
  SQL Injection        |      348 |    1,391 |      80.0%
  XSS                  |        0 |      232 |     100.0%
  Path Traversal       |        0 |       33 |     100.0%
  Command Injection    |        0 |       34 |     100.0%
  CRLF Injection       |        0 |      249 |     100.0%


In [7]:
# %% [2] 개별 키워드 상세 분석 (공격 빈도 Top 20)
# ============================================================
# 2. 개별 키워드 상세 분석
# ============================================================
print(f"\n{'─' * 65}")
print("  [2] 개별 키워드 상세 빈도 (공격에서 많이 발견된 순)")
print(f"{'─' * 65}")

results = []
for category, keywords in attack_categories.items():
    for kw in keywords:
        n_count = normal_text.str.contains(re.escape(kw), case=False, na=False).sum()
        a_count = attack_text.str.contains(re.escape(kw), case=False, na=False).sum()
        results.append({
            "category": category, "keyword": kw,
            "normal": n_count, "attack": a_count,
        })

result_df = pd.DataFrame(results).sort_values("attack", ascending=False)

print(f"\n  {'키워드':<15s} | {'카테고리':<20s} | {'정상':>6s} | {'공격':>6s}")
print(f"  {'─' * 15}-+-{'─' * 20}-+-{'─' * 6}-+-{'─' * 6}")

for _, row in result_df.head(20).iterrows():
    marker = " ★" if row["attack"] > row["normal"] * 2 else ""
    print(f"  {row['keyword']:<15s} | {row['category']:<20s} | "
          f"{row['normal']:>6,} | {row['attack']:>6,}{marker}")

print(f"\n  ★ = 공격에서 정상보다 2배 이상 많이 발견된 키워드")



─────────────────────────────────────────────────────────────────
  [2] 개별 키워드 상세 빈도 (공격에서 많이 발견된 순)
─────────────────────────────────────────────────────────────────

  키워드             | 카테고리                 |     정상 |     공격
  ───────────────-+-────────────────────-+-──────-+-──────
  '               | SQL Injection        |     19 |    518 ★
  insert          | SQL Injection        |    327 |    437
  --              | SQL Injection        |      0 |    309 ★
  %0a             | CRLF Injection       |      0 |    156 ★
  <script         | XSS                  |      0 |    106 ★
  alert(          | XSS                  |      0 |    100 ★
  %0d             | CRLF Injection       |      0 |     93 ★
  select          | SQL Injection        |      0 |     37 ★
  drop            | SQL Injection        |      2 |     35 ★
  |               | Command Injection    |      0 |     34 ★
  /etc/passwd     | Path Traversal       |      0 |     33 ★
  1=1             | SQL Injection        |  

In [8]:
# %% [3] URL 인코딩 패턴 분석
# ============================================================
# 3. URL 인코딩 패턴 분석
# ============================================================
print(f"\n{'─' * 65}")
print("  [3] URL 인코딩 패턴 분석 (인코딩 전 원본 URL 기준)")
print(f"{'─' * 65}")

encoded_patterns = {
    "%27": "'  (작은따옴표 — SQL Injection)",
    "%3C": "<  (꺾쇠 — XSS)",
    "%3E": ">  (꺾쇠 닫기 — XSS)",
    "%2F": "/  (슬래시 — Path Traversal)",
    "%2E": ".  (점 — Path Traversal)",
    "%0D": "CR (캐리지리턴 — CRLF Injection)",
    "%0A": "LF (라인피드 — CRLF Injection)",
    "%20": "   (공백 — SQL 키워드 구분)",
    "%00": "NULL (널바이트 — 우회 시도)",
}

print(f"\n  {'패턴':<6s} | {'의미':<35s} | {'정상':>6s} | {'공격':>6s}")
print(f"  {'─' * 6}-+-{'─' * 35}-+-{'─' * 6}-+-{'─' * 6}")

# 인코딩 전 원본 URL에서 검색
normal_url = df[df["label"] == "Normal"]["url"]
attack_url = df[df["label"] == "Anomalous"]["url"]

for pattern, meaning in encoded_patterns.items():
    n_count = normal_url.str.contains(pattern, case=False, na=False).sum()
    a_count = attack_url.str.contains(pattern, case=False, na=False).sum()
    if n_count + a_count > 0:
        print(f"  {pattern:<6s} | {meaning:<35s} | {n_count:>6,} | {a_count:>6,}")



─────────────────────────────────────────────────────────────────
  [3] URL 인코딩 패턴 분석 (인코딩 전 원본 URL 기준)
─────────────────────────────────────────────────────────────────

  패턴     | 의미                                  |     정상 |     공격
  ──────-+-───────────────────────────────────-+-──────-+-──────
  %27    | '  (작은따옴표 — SQL Injection)          |      6 |    240
  %3C    | <  (꺾쇠 — XSS)                       |      0 |     94
  %3E    | >  (꺾쇠 닫기 — XSS)                    |      0 |     94
  %2F    | /  (슬래시 — Path Traversal)           |    125 |    394


In [9]:
# %% [4] 접근 경로(Path) 분석
# ============================================================
# 4. 접근 경로(Path) 분석
# ============================================================
print(f"\n{'─' * 65}")
print("  [4] 가장 많이 접근된 경로 (정상 vs 공격)")
print(f"{'─' * 65}")

df["path"] = df["url_decoded"].str.split("?").str[0]

print(f"\n  === 정상 요청 Top 5 ===")
normal_paths = df[df["label"] == "Normal"]["path"].value_counts().head(5)
for path, count in normal_paths.items():
    print(f"    {path:<50s} {count:>6,}건")

print(f"\n  === 공격 요청 Top 5 ===")
attack_paths = df[df["label"] == "Anomalous"]["path"].value_counts().head(5)
for path, count in attack_paths.items():
    print(f"    {path:<50s} {count:>6,}건")



─────────────────────────────────────────────────────────────────
  [4] 가장 많이 접근된 경로 (정상 vs 공격)
─────────────────────────────────────────────────────────────────

  === 정상 요청 Top 5 ===
    /tienda1/publico/entrar.jsp                           367건
    /tienda1/publico/registro.jsp                         352건
    /tienda1/publico/vaciar.jsp                           348건
    /tienda1/miembros/editar.jsp                          340건
    /tienda1/publico/autenticar.jsp                       335건

  === 공격 요청 Top 5 ===
    /tienda1/publico/registro.jsp                         490건
    /tienda1/publico/pagar.jsp                            464건
    /tienda1/miembros/editar.jsp                          460건
    /tienda1/publico/autenticar.jsp                       447건
    /tienda1/publico/anadir.jsp                           444건


In [10]:
# %% [5] URL 길이 비교 (정상 vs 공격)
# ============================================================
# 5. URL 길이 비교
# ============================================================
print(f"\n{'─' * 65}")
print("  [5] URL 길이 비교 (정상 vs 공격)")
print(f"{'─' * 65}")

df["url_length"] = df["url_decoded"].str.len()

for label in ["Normal", "Anomalous"]:
    subset = df[df["label"] == label]["url_length"]
    print(f"\n  {label}:")
    print(f"    평균: {subset.mean():.0f},  중앙값: {subset.median():.0f},  "
          f"최대: {subset.max():,},  표준편차: {subset.std():.0f}")

print(f"\n  >> 공격 요청의 URL이 평균적으로 더 깁니다!")
print(f"     이 차이가 8주차 IQR 이상치 탐지의 근거가 됩니다.")


print(f"\n{'=' * 65}")
print("  키워드 분석 완료!")
print("  Streamlit 대시보드로 시각적 탐색: streamlit run eda_visualization.py")
print(f"{'=' * 65}")



─────────────────────────────────────────────────────────────────
  [5] URL 길이 비교 (정상 vs 공격)
─────────────────────────────────────────────────────────────────

  Normal:
    평균: 50,  중앙값: 29,  최대: 328,  표준편차: 60

  Anomalous:
    평균: 73,  중앙값: 32,  최대: 637,  표준편차: 84

  >> 공격 요청의 URL이 평균적으로 더 깁니다!
     이 차이가 8주차 IQR 이상치 탐지의 근거가 됩니다.

  키워드 분석 완료!
  Streamlit 대시보드로 시각적 탐색: streamlit run eda_visualization.py
